In [ ]:
from pathlib import Path
from tqdm import tqdm

miv_ecg_root = Path("MIMIC-IV-data\\mimic-iv-ecg-matched-subset\\mimic-iv-ecg_complete\\files")
num_groups = 0
num_subjects = 0
num_studies = 0

missing_hea = []
missing_dat = []

missing_studies = []

for gr in miv_ecg_root.iterdir():
    if gr.name in ["index.html", "nothing.txt", "RECORDS"]: continue
    num_groups+=1
    for sb in gr.iterdir():
        if sb.name in ["index.html", "nothing.txt", "RECORDS"]: continue
        num_subjects+=1
        for st in sb.iterdir():
            if st.name == "index.html": continue
            num_studies+=1
            inner_name = str(st.name[1:])
            dat_file = st / (inner_name+".dat")
            hea_file = st / (inner_name+".hea")
            missing_study = False
            if not dat_file.exists(): 
                missing_dat.append(dat_file)
                missing_study = True
            if not hea_file.exists(): 
                missing_hea.append(hea_file)
                missing_study = True
            if missing_study:
                missing_studies.append(Path(*st.parts[-4:]) / inner_name)


print(f"num_groups: {num_groups}, num_subjects: {num_subjects}, num_studies: {num_studies}")
print(f"{len(missing_hea)} missing hea files, {len(missing_dat)} missing dat files")
#num_groups: 1000, num_subjects: 161210, num_studies: 798413

#approx 62.2Kb for one 300x300 ECG image, totals us at ~47Gb
#currently the .hea and .dat files take up 90Gb
#2063 missing hea files, 2061 missing dat files

ms_file = miv_ecg_root.parent / "missing_studies.txt"
with open(ms_file, 'w') as f:
    for line in missing_studies:
        f.write(str(line) + '\n')
mh_file = miv_ecg_root.parent / "missing_headers.txt"
with open(mh_file, 'w') as f:
    for line in missing_hea:
        f.write(str(line) + '\n')
md_file = miv_ecg_root.parent / "missing_dat.txt"
with open(md_file, 'w') as f:
    for line in missing_dat:
        f.write(str(line) + '\n')

num_groups: 1000, num_subjects: 161210, num_studies: 798413
2063 missing hea files, 2061 missing dat files


In [ ]:
import os
import requests

# Replace this with your list of file paths (relative to the base URL)
file_paths = [
    "path/to/file1.ext",
    "path/to/file2.ext",
    # Add all other paths here
]

# Base URL where the files are hosted
base_url = "https://physionet.org/files/mimic-iv-ecg/1.0/"
# Local destination directory where you want to save files
destination_dir = "C:/path/to/save/directory"

# Function to download a file and save it to the correct path
def download_file(file_path):
    # Construct the full URL and the local save path
    url = f"{base_url}{file_path}"
    save_path = os.path.join(destination_dir, file_path)

    # Create directories if they don't exist
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Download the file
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Raise an error for bad status codes
        
        # Write to file in chunks
        with open(save_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)

        print(f"Downloaded: {file_path}")

    except requests.exceptions.RequestException as e:
        print(f"Failed to download {file_path}: {e}")

# Loop over the list of file paths and download each one
for path in file_paths:
    download_file(path)
